# Week 5 Lab: Teaching a Machine a Law of Nature
**The Evolution of Machine Learning and AI · SETU**

> **How this lab works.** Six exercises, one per lecture part, in the same order. In each one you
> **predict** before you run, **run** the experiment, **look** at the right column, and **explain**
> the result in your own words. In Exercises 2 and 4 you also **build** the parts that make a
> physics-informed network work: by the end of Exercise 4, everything except the network and the
> training loop from Week 3 is your own code. Most exercises end with **one thing to change**.
>
> **`SUBMISSION.md` is blank.** Write your answers there as you go. A wrong prediction is not a
> mistake: it is the most useful line on the sheet, because it shows you exactly what to rethink.
> The printouts tell you what was measured and which column to read; what it means is your part.
>
> **The reference answers are a separate card on Tutors, *Week 5 Reference Answers*.** Open it only
> after you have uploaded `SUBMISSION.md`, and compare it with what you wrote.
>
> **The `check_my_...` cells test your code** and tell you what to look at if it is not right yet.
> They never print the answer. **If anything takes more than 40 minutes, stop and post in the
> channel.** That is a bug in the material, not in you.
>
> **Cells that train are slower than usual** (20 to 90 seconds), because checking an equation means
> taking slopes of the network inside every training step. Each step says how long.

| Exercise | Lecture part | Slides | You build | Time |
|---|---|---|---|---|
| 1 · A law is not a pattern | Part 1 | 7 to 10 | | 20 min |
| 2 · Checking the equation where there is no data | Part 2 | 13 to 17 | the residual and the physics loss | 35 min |
| 3 · What the law buys | Part 3 | 21 to 24 | | 20 min |
| 4 · What it costs, and how it goes wrong | Part 4 | 27 to 34 | the hard constraint | 35 min |
| 5 · Four more ways to build in science | Part 5 | 37 to 43 | | 25 min |
| 6 · Choosing | Part 6 | 46 to 49 | | 20 min |
| | Last two questions, then upload | | | 10 min |

About 2 h 45 min in total, or about 2 h 25 min if you skip the steps marked *optional*.

**Run the cell below first.** It loads the lab code.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 60)

import lecture_steps as ls      # one helper per lecture idea; you never edit it
from autodiff import mean       # the average, inside the engine (you will need it in Exercise 2)
print("Lab code loaded. You are ready for Exercise 1.")

Lab code loaded. You are ready for Exercise 1.


---
# Exercise 1 · A law is not a pattern  *(≈ 20 min)*

### From the lecture
*Part 1, slides 7 to 10.* **Red box 1:** *A pattern in data is a nudge: usually right, eliminates nothing. A law of nature is a constraint: exactly true, everywhere, and it removes whole sets of answers. A model told the law can score better with no data at all than a model shown examples.*

### What you are doing
**In one sentence:** you will race a network that is told the law of a swinging spring, and shown
no measurements at all, against the same network shown up to 64 measurements.

### Predict first  *(SUBMISSION.md, 1a and 1b)*
- **1a.** Before running anything: write one *pattern* and one *law* from your own work or life.
- **1b.** In the race, which model ends with the lower error: data-only with 64 measurements, or
  physics-informed with none? By roughly what factor: 2, 20 or 200 times? Commit to an answer and
  one reason.

### Run: two kinds of advice

In [2]:
ls.two_kinds_of_advice()

Printout A: two kinds of advice, and what each one does to the list of possible answers
  A pattern (Week 4):  'people in high-visibility jackets are often construction workers'
                       -> a nudge. Usually right. Eliminates nothing; every answer stays possible.
  A law (this week):   'this person is standing in Dublin, so they are not in Cork'
                       -> a constraint. Removes a whole set of answers, permanently, whatever new evidence arrives.
  Week 4 gave models the first kind. This week gives them the second.


### Look at
**Printout A.** Does your pattern only *nudge*, and does your law *remove* answers? If not, fix
1a now and say what you changed.

### Run: the race  *(about 70 seconds)*

In [3]:
ls.the_race()

Table B: a damped spring. Every row is the SAME network with the SAME optimiser for the SAME number of steps.
         Read the observations column and the test RMSE column (lower is better).


,model,observations,parameters,test RMSE
0,physics-informed (no data),0,2209,0.00135
1,data-only,4,2209,0.31289
2,data-only,8,2209,0.06556
3,data-only,16,2209,0.04326
4,data-only,32,2209,0.02962
5,data-only,64,2209,0.02648
6,physics + 8 noisy points,8,2209,0.02665


### Look at
**Table B**: the `observations` column and the `test RMSE` column (lower is better).

### Explain  *(SUBMISSION.md, 1c and 1d)*
- **1c.** Record the two errors, and say whether your prediction in 1b was right.
- **1d.** In your own words: what did each model's loss compare the network's guess *against*?
  Use that to explain why one model needs measurements and the other does not.

### Change one thing  *(optional, 3 min; SUBMISSION.md, 1e)*
Give the data-only model **perfect** measurements, with no noise at all. **Predict first:** with 64
perfect measurements, will it now beat the physics-informed model? Then run:

In [4]:
ls.the_race_without_noise()

Table B2: the data-only rows of Table B again, with every measurement exact (noise 0).
          Compare the test RMSE with the physics-informed row of Table B, which saw no measurements at all.


,model,observations,parameters,test RMSE
0,"data-only, noise-free measurements",16,2209,0.00694
1,"data-only, noise-free measurements",64,2209,0.00530


**Table B2.** Was your prediction right? Finish 1e: if noise was not the whole story, where does
the data-only model's error come from?

---
# Exercise 2 · Checking the equation where there is no data  *(≈ 35 min; the code on slide 18)*

### From the lecture
*Part 2, slides 13 to 17.* **Red box 2:** *A physics-informed network is an ordinary network with an unusual loss: put the network's guess back into the equation and square what is left over. Checking an equation does not need the answer. The one new tool is taking the derivative of a derivative, which the engine does exactly.*

### What you are doing
**In one sentence:** you will write the two lines that turn an ordinary network into a
physics-informed one, mark homework with them, and then train a network with them.

### Predict first  *(SUBMISSION.md, 2a)*
- **2a.** Three candidate answers for the spring: the true solution; `cos(t)`, a swing that forgets
  the friction; and a flat line at 0.7. Each is put back into  m u'' + c u' + k u. Which leaves
  exactly zero? Which leaves the most?

### Build 1: the leftover  *(SUBMISSION.md, 2b)*
`u` is the guess, `u_t` its slope, and `u_tt` the slope of its slope, at many times at once.
`m`, `c`, `k` are the spring's numbers. Return what is left over when the guess is put back into
**m u'' + c u' + k u = 0**. One line, using only `+`, `-` and `*`, so the same line works on plain
numbers and on the engine's values. Run the cell until every check shows ✓.

In [5]:
def my_residual(u, u_t, u_tt, m, c, k):
    """What is left over when a guess is put back into  m u'' + c u' + k u = 0."""
    return None      # <- replace None with your one line

ls.check_my_residual(my_residual)

Check: my_residual
  ✗ it returns None. Replace `return None` with your one line.


### Run: mark the homework with your residual

In [6]:
ls.mark_the_homework(my_residual)

my_residual still returns None: write it first (Exercise 2, Build 1), and run its check cell until every line shows ✓.


### Look at
**Printout A.** Was 2a right? Then answer **2c**: the flat line leaves the same number at every
time. Which number, and why exactly that one? *(Work out u' and u'' for a flat line first.)*

### Build 2: the physics loss  *(SUBMISSION.md, 2d)*
Turn the leftover at 200 times into **one number** that is zero only when the leftover is zero
everywhere. The leftover can be positive at one time and negative at another. You have `mean(...)`
from the first cell; `*` and `**` work too. One line.

In [7]:
def my_physics_loss(residual):
    """One number: zero only when the residual is zero at every time."""
    return None      # <- replace None with your one line

ls.check_my_loss(my_physics_loss)

Check: my_physics_loss
  ✗ it returns None. Replace `return None` with your one line.


### Run: train a network with your two functions  *(about 20 seconds)*
No measurements. The training loop is the one from Week 3; the printout shows where your code
goes into it.

In [8]:
ls.train_with_my_loss(my_residual, my_physics_loss)

my_residual still returns None: write it first (Exercise 2, Build 1), and run its check cell until every line shows ✓.


### Look at
**The plot** and **Table G**: compare your test RMSE with the physics-informed row of Exercise 1,
Table B.

### Explain  *(SUBMISSION.md, 2e and 2f)*
- **2e.** Record your error and the Table B one. What did your two lines have to get right for
  them to match?
- **2f.** *Write this before running the next cell.* Week 3's loop was guess, check, blame,
  nudge. Which step did your code replace, and why does that step not need any measurements?
  Then run the cell and compare.

In [9]:
ls.one_thing_changed()

Table D: what changed between Weeks 3 and 4 and this week


,component,Weeks 3 and 4,Week 5
0,the network,feedforward,the same
1,forward pass,inputs to outputs,the same
2,the loss,distance to labels,residual of an equation <- the only change
3,backward pass,d loss / d weights,the same
4,optimiser,Adam,the same
5,the loop,"guess, check, blame, nudge",the same


### Optional: why the engine, and not finite differences  *(3 min; SUBMISSION.md, 2g)*

In [ ]:
ls.the_engine()
ls.why_not_finite_differences()

**Printouts B and C.** For 2g: which derivative does the finite difference get more wrong, and
why does that matter for a loss built on u''?

---
# Exercise 3 · What the law buys  *(≈ 20 min)*

### From the lecture
*Part 3, slides 21 to 24.* **Red box 3:** *A model that knows the law keeps working beyond the data, because the law was known there too. It also inherits every consequence of the law, such as conservation of energy, whether or not you thought to check. A model fitted to points inherits only agreement with those points.*

### What you are doing
**In one sentence:** you will give both models measurements for only the first 40% of the time
and see what happens after that, then remove the friction and see what happens to energy.

### Predict first  *(SUBMISSION.md, 3a)*
- **3a.** Both models get measurements only for t from 0 to 4 (the run lasts 10). The physics
  model also checks the equation over the whole run. For each model, predict the *degradation
  factor* (its error beyond the data divided by its error where data existed): about 1, about 3,
  about 10, or about 30?

### Run: beyond the data  *(about 25 seconds)*

In [10]:
ls.beyond_the_data()

Table A: measurements on the first 40% of the time only (t from 0 to 4); scored on the rest.
         degradation factor = error beyond the data / error where data existed. 1.0 means no worse.


,model,RMSE where data existed,RMSE beyond the data,degradation factor
0,data-only,0.0181,0.4998,27.6
1,physics + same data,0.0086,0.0087,1.0


### Look at
**Table A**, the last column.

### Explain  *(SUBMISSION.md, 3b and 3c)*
- **3b.** Record both factors and compare them with 3a.
- **3c.** In your own words: why does the physics model keep working where it never saw data?

### Change one thing  *(optional, 3 min; SUBMISSION.md, 3d)*
Move the edge of the data back to 20% of the run. **Predict first:** whose degradation factor
changes more, and in which direction? Then run:

In [11]:
ls.beyond_the_data(t_split=2.0)

Table A: measurements on the first 20% of the time only (t from 0 to 2); scored on the rest.
         degradation factor = error beyond the data / error where data existed. 1.0 means no worse.


,model,RMSE where data existed,RMSE beyond the data,degradation factor
0,data-only,0.0128,0.8174,63.9
1,physics + same data,0.0032,0.0049,1.5


### Predict first  *(SUBMISSION.md, 3e)*
- **3e.** Now the friction is removed, so the spring's energy should stay exactly constant. *Energy
  is in nobody's loss.* Rank the three models (data-only; physics plus data; physics only, no data)
  from most energy drift to least.

### Run: energy nobody asked for  *(about 45 seconds)*

In [12]:
ls.energy_nobody_asked_for()

Table B: remove the friction, so energy should stay exactly constant. Nobody told any model this.
         Read the last column: how far each model's energy drifts from the truth.


,model,energy at t=0,energy at t=T,max drift from true,drift as % of true
0,data-only,0.52,0.4189,0.0816,16.3%
1,physics + data,0.50,0.4865,0.0254,5.1%
2,"physics only, no data",0.50,0.4995,0.0012,0.2%


### Look at
**Table B**, the last column.

### Explain  *(SUBMISSION.md, 3f)*
- **3f.** Record the drift for data-only and for physics only. Energy was never in a loss: how did
  the physics-only model keep it constant?

---
# Exercise 4 · What it costs, and the two ways it goes quietly wrong  *(≈ 35 min; the code on slide 31)*

### From the lecture
*Part 4, slides 27 to 34.* **Red box 4:** *A soft penalty needs a weight nobody can derive, and a bad weight gives a bad model that still satisfies the condition. A hard constraint builds the condition into the arithmetic and deletes the problem. A small loss only speaks for the places you checked. And a wrong law is worse than no law: the loss stays small while the answer goes wrong.*

### What you are doing
**In one sentence:** you will try two ways of building in the starting condition, build the
better one yourself, then find out what a small loss does and does not promise, and what a wrong
law does.

### Predict first  *(SUBMISSION.md, 4a)*
- **4a.** The start u(0) = 1 is asked for with a *soft penalty* of weight 1, 50 or 500 (bigger
  means "this matters more"). Which weight gives the lowest error? Will the weight-500 model get
  u(0) = 1 right?

### Run: the weight nobody can derive  *(about 90 seconds)*

In [13]:
ls.the_weight_nobody_can_derive()

Table A: the starting condition u(0) = 1 imposed four ways. Read BOTH the u(0) column and the error column.


,how the initial condition is imposed,"u(0), should be 1","u'(0), should be 0",test RMSE
0,"soft penalty, weight 1",0.99988,-0.00005,0.00154
1,"soft penalty, weight 50",0.99941,-0.00019,0.32225
2,"soft penalty, weight 500",0.99953,-0.00048,0.66444
3,hard constraint,1.00000,0.00000,0.00135


### Look at
**Table A**: the `u(0)` column *and* the `test RMSE` column.

### Explain  *(SUBMISSION.md, 4b)*
- **4b.** How can the weight-500 model get the start right and still be badly wrong?

### Build 3: the hard constraint  *(SUBMISSION.md, 4c)*
`N` is the network's raw output at times `t`. Return a guess `u` built from `N` so that
**u(0) = 1 and u'(0) = 0 for any network at all**, before any training. Use `t`, `N`, numbers and
`+ - * /`. Hints, one at a time if you need them: (1) start from `1 + something * N`; (2) the
something must be zero at t = 0; (3) so must its slope.

In [14]:
def my_guess(t, N):
    """A guess with u(0) = 1 and u'(0) = 0 built in, whatever N is."""
    return None      # <- replace None with your guess

ls.check_my_constraint(my_guess)

Check: my_guess
  ✗ it returns None. Replace `return None` with your guess.


### Run: train with all three of your functions  *(about 20 seconds)*

In [15]:
ls.train_with_my_constraint(my_guess, my_residual, my_physics_loss)

my_guess still returns None: write it first (Exercise 4, Build 3), and run its check cell until every line shows ✓.


### Look at
**The plot** and **Table A2**: compare your error with the `hard constraint` row of Table A.

### Explain  *(SUBMISSION.md, 4d)*
- **4d.** Record your error. If it is close to the reference, say why your guess works. If it is
  much worse, use the printout's clue to explain why, and (optional) try a second version.

### Predict first  *(SUBMISSION.md, 4e)*
- **4e.** The equation is checked at 10, 50 or 200 places (checkpoints). Which run will *report*
  the smallest physics loss? Which will have the smallest *real* error?

### Run: too few checkpoints  *(about 40 seconds)*

In [16]:
ls.too_few_checkpoints()

Table B: the same network trained with the equation checked at 10, 50 and 200 places (the checkpoints).
         Middle column: the physics loss the model reports. Last column: its real error against the truth.


,checkpoints (collocation points),physics loss it reports,true error (RMSE)
0,10,2.61e-06,0.2983
1,50,4.94e-05,0.0015
2,200,8.66e-05,0.0035


### Look at
**Table B**: the middle column against the last column.

### Explain  *(SUBMISSION.md, 4f)*
- **4f.** What does a small physics loss actually promise, and what does it leave unchecked?

### Change one thing: your own wrong law  *(SUBMISSION.md, 4g)*
Question 6 of the lecture's seven: *what happens if my law is wrong by 20%?* The true friction is
0.2; `0.24` below is 20% too high. Keep it or choose your own value. **Predict first:** told your
value, will the physics model beat the data-only model, match it, or lose to it? Then run
*(about 90 seconds)*:

In [ ]:
my_wrong_c = 0.24          # <- the friction you tell the model (the truth is 0.2)
ls.when_the_law_is_wrong(c_assumed=(0.2, my_wrong_c, 0.4, 0.8))

### Look at
**Table C**: the `true RMSE` column, then the `physics loss it reports` column.

### Explain  *(SUBMISSION.md, 4h)*
- **4h.** Was 4g right? At c = 0.8, would the reported loss have warned you? In your own words:
  why is a wrong law worse than no law?

---
# Exercise 5 · Four more ways to build in science  *(≈ 25 min; the code on slide 40)*

### From the lecture
*Part 5, slides 37 to 43.* **Red box 5:** *Symmetry: feed the model only what cannot change under the symmetry, and it cannot get it wrong; but a false symmetry is a wall. A neural ODE learns the rule, not the answer, so it can start anywhere. A neural operator learns a whole family at once. Diffusion borrows its forward process from physics and learns only one small reversal.*

### What you are doing
**In one sentence:** four quick predictions, four runs, then explain each idea in one sentence of
your own.

### Predict first  *(SUBMISSION.md, 5a)*
- **5a.** Three particles; their energy depends only on the distances between them. One model reads
  raw coordinates, one reads the three distances. Rotate the whole arrangement: whose answer
  changes, and by roughly how much?

In [ ]:
display(ls.spin_it())
display(ls.unseen_orientations())

### Look at
**Tables A and B.** Record in **5b**.

### Predict first  *(SUBMISSION.md, 5c)*
- **5c.** Add gravity, so which way up the arrangement sits now changes the energy. Which model
  wins now? Would more data rescue the other one?

In [ ]:
ls.a_false_symmetry()

### Look at
**Table C.** Explain in **5d**: why can the losing model never catch up?

### Predict first  *(SUBMISSION.md, 5e)*
- **5e.** A model trained on a window of time is run three times further. One model learned the
  *answer* (position at each time); one learned the *rule* (how position changes). Which degrades
  more?

### Run: learn the rule, and solve a whole family  *(about 40 seconds)*

In [ ]:
display(ls.learn_the_rule())
display(ls.a_whole_family())

### Look at
**Tables D and E.** Record in **5f**, and say what the operator trades for its speed.

### Run: ink in water

In [ ]:
ls.ink_in_water()

### Look at
**Table F**, the last column. Nothing was trained here.

### Explain: teach it back  *(SUBMISSION.md, 5g)*
- **5g.** One sentence each, in your own words, for a colleague who missed the lecture: symmetry,
  a neural ODE, a neural operator, and diffusion (say which half is physics and which half is
  learned).

---
# Exercise 6 · Choosing  *(≈ 20 min)*

### From the lecture
*Part 6, slides 46 to 49.* **Red box 6:** *Match the method to what you know, not to the method's fame. Physics-informed learning is not a better way to solve a standard equation; a classical solver is. It is a better way to combine sparse data with a known law, infer unknown numbers, and build fast surrogates. Seven questions before you build.*

### What you are doing
**In one sentence:** one last prediction on a harder problem, then you match situations to methods
yourself before seeing the reference, and apply the seven questions to a problem of your own.

### Predict first  *(SUBMISSION.md, 6a)*
- **6a.** Heat spreading along a rod: two inputs (position and time) instead of one. Physics-informed
  with 0 measurements, or data-only with 128: which wins, and by roughly how much?

### Run: a PDE with no data  *(about 60 seconds)*

In [ ]:
ls.a_pde_with_no_data()

### Look at
**Table A.** Record in **6b**.

### Match first, then reveal  *(SUBMISSION.md, 6c)*
Run the cell, write your seven matches in 6c, **then** run the reveal cell and mark them.

In [ ]:
ls.match_the_method()

**Only once your matches are written down:**

In [ ]:
ls.match_the_method(reveal=True)

### Apply  *(SUBMISSION.md, 6d)*

In [ ]:
ls.seven_questions()

- **6d.** Pick a problem you know (your work, a hobby, or your capstone idea) where some rule is
  known. Answer questions 1, 2, 6 and 7 for it, one sentence each.

---
# Before you upload  *(10 min)*
- **R1.** Which prediction were you most wrong about? What had you assumed, and what do you think now?
- **R2.** Roughly how long did the lab take?

Add your name, upload `SUBMISSION.md` (see the *Submitting* page on Tutors), **then** open the
*Week 5 Reference Answers* card and compare. Where the reference explains something differently
from you, it is worth one more line in your own notes.

---
# Optional: going further  *(not required · only if you want more)*

- **The browser toys.** `interactive/pinn-sandbox.html` trains a physics-informed network in your browser; tab 3 shows a confidently wrong model. `symmetry-explorer.html`: press Spin. `diffusion-thermodynamics-explorer.html`: run the ink forward and back.
- **Run it backwards.** Copy `my_residual` and make the friction `c` something the network learns from a few noisy measurements: an *inverse problem* (Table B row 4 of Exercise 6). Predict how many measurements you need first.
- **Break it on purpose.** Replace tanh with ReLU in `mlp.py` and run `ls.train_with_my_loss` again; explain the failure in one sentence (think about u'').
- **The full run.** `python run_lab.py` reproduces every number in the lecture (several minutes; `--only pinn,heat` for part of it).